# Matrix Computation Examples

This notebook demonstrates practical matrix computations using the chromatic matroids package, based on the examples in the package repository.

Make sure you create a new virtual environment and install the package before running the code.


In [ ]:
# Install the package from local sources (run once per environment)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "../package", "--quiet"])

In [1]:
from chromatic_matroids import (
    generate_loopless_nested_matroids,
    generate_loopless_schubert_matroids,
    chromatic_quasisymmetric_function,
    chromatic_non_commutative_quasisymmetric_function,
    SetComposition,
    NCQSymFunction,
    QSymFunction,
    compute_conjecture_matrix,
    compute_lowerbound_matrix,
    compute_conjecture_big_matrix,
    compute_conjecture_alternatingsum_matrix
)
import numpy as np
import time
from math import factorial

print("✓ All imports successful!")

ModuleNotFoundError: No module named 'chromatic_matroids'

## Example 1: Basic Rank Computation for Loopless Nested Matroids

This example is based on `experimentation.py`. We compute the chromatic non-commutative quasisymmetric functions for all loopless nested matroids of a given size and analyze the rank of the resulting matrix.

In [2]:
def compute_ncqsym_matrix_rank(dimension):
    """
    For a given dimension, compute chromatic NCQSym functions for all loopless nested matroids
    and return the rank of the resulting matrix.
    """
    # Generate all loopless nested matroids
    loopless_nested = generate_loopless_nested_matroids(dimension)
    
    # Compute chromatic NCQSym functions
    wqsym = [chromatic_non_commutative_quasisymmetric_function(matroid) 
             for matroid in loopless_nested]
    
    # Collect all unique set composition indices
    set_coefs = set()
    for symfunc in wqsym:
        for key in symfunc.coefficients:
            set_coefs.add(key)
    
    # Build matrix: rows = matroids, columns = set compositions
    mat_coeffs = [[0 for _ in set_coefs] for _ in wqsym]
    for i, symfunc in enumerate(wqsym):
        for j, key in enumerate(set_coefs):
            mat_coeffs[i][j] = symfunc.coefficients.get(key, 0)
    
    # Compute rank
    rank = np.linalg.matrix_rank(mat_coeffs)
    
    return {
        'dimension': dimension,
        'num_matroids': len(loopless_nested),
        'num_set_compositions': len(set_coefs),
        'matrix_shape': (len(wqsym), len(set_coefs)),
        'rank': rank,
        'wqsym_functions': wqsym
    }

# Test with dimension 4
print("Computing matrix rank for loopless nested matroids...\n")
result = compute_ncqsym_matrix_rank(4)

print(f"Dimension: {result['dimension']}")
print(f"Number of loopless nested matroids: {result['num_matroids']}")
print(f"Number of unique set compositions: {result['num_set_compositions']}")
print(f"Matrix shape: {result['matrix_shape']}")
print(f"Matrix rank: {result['rank']}")

Computing matrix rank for loopless nested matroids...

Dimension: 4
Number of loopless nested matroids: 24
Number of unique set compositions: 75
Matrix shape: (24, 75)
Matrix rank: 24


## Example 2: Dimension Computation Analysis

This example is based on `dimension_computation.py`. We compute ranks of both commutative and non-commutative quasisymmetric function matrices for multiple dimensions and compare them to theoretical values.

In [3]:
def compute_stable_matroid_rank(dimension, verbose=True):
    """
    Compute stable matroid scores and analyze ranks for QSym and NCQSym.
    This is computationally intensive for larger dimensions.
    """
    if verbose:
        print(f"\nProcessing dimension d={dimension}...")
    
    # Generate loopless nested matroids
    if verbose:
        print("  Creating loopless nested matroids...")
    loopless_nested = generate_loopless_nested_matroids(dimension)
    
    # Generate set compositions
    if verbose:
        print("  Creating set compositions...")
    setcompositions = SetComposition.generate_all_setcompositions(dimension)
    
    # Initialize NCQSym functions
    if verbose:
        print("  Computing stable matroid set compositions...")
    ncqsym = [NCQSymFunction() for _ in loopless_nested]
    
    for matroid_index, matroid in enumerate(loopless_nested):
        for sc_index, sc in enumerate(setcompositions):
            # Check if this set composition is "stable" for this matroid
            from chromatic_matroids import stable_matroids_setcompositions
            if stable_matroids_setcompositions(matroid, sc):
                ncqsym[matroid_index] += NCQSymFunction(monomial_basis=sc)
    
    # Convert to commutative QSym by calling comu()
    if verbose:
        print("  Converting to commutative QSym...")
    qsym = [nc.comu() for nc in ncqsym]
    
    # Compute ranks
    def compute_matrix_rank(sym_functions):
        all_keys = set()
        for func in sym_functions:
            all_keys.update(func.coefficients.keys())
        
        matrix = [[func.coefficients.get(key, 0) for key in sorted(all_keys)] 
                  for func in sym_functions]
        return np.linalg.matrix_rank(matrix) if matrix else 0
    
    rank_qsym = compute_matrix_rank(qsym)
    rank_ncqsym = compute_matrix_rank(ncqsym)
    
    results = {
        'dimension': dimension,
        'num_matroids': len(loopless_nested),
        'num_set_compositions': len(setcompositions),
        'rank_qsym': rank_qsym,
        'expected_qsym': 2**(dimension-1),
        'rank_ncqsym': rank_ncqsym,
        'expected_ncqsym': factorial(dimension)
    }
    
    if verbose:
        print(f"  Number of set compositions: {results['num_set_compositions']}")
        print(f"  QSym rank: {results['rank_qsym']} (expected: {results['expected_qsym']})")
        print(f"  NCQSym rank: {results['rank_ncqsym']} (expected: {results['expected_ncqsym']})")
    
    return results

# Test with small dimensions
print("=" * 60)
print("DIMENSION ANALYSIS: Stable Matroid Ranks")
print("=" * 60)

results_list = []
for d in range(2, 5):  # Test dimensions 2-4 (5 is slower)
    result = compute_stable_matroid_rank(d)
    results_list.append(result)

print("\n" + "=" * 60)
print("Summary Table")
print("=" * 60)
print(f"{'d':<3} {'#Matroids':<12} {'#SetComp':<12} {'Rank QSym':<15} {'Rank NCQSym':<15}")
print("-" * 60)
for r in results_list:
    print(f"{r['dimension']:<3} {r['num_matroids']:<12} {r['num_set_compositions']:<12} "
          f"{r['rank_qsym']:<15} {r['rank_ncqsym']:<15}")

DIMENSION ANALYSIS: Stable Matroid Ranks

Processing dimension d=2...
  Creating loopless nested matroids...
  Creating set compositions...
  Computing stable matroid set compositions...
  Converting to commutative QSym...
  Number of set compositions: 3
  QSym rank: 2 (expected: 2)
  NCQSym rank: 2 (expected: 2)

Processing dimension d=3...
  Creating loopless nested matroids...
  Creating set compositions...
  Computing stable matroid set compositions...
  Converting to commutative QSym...
  Number of set compositions: 13
  QSym rank: 4 (expected: 4)
  NCQSym rank: 6 (expected: 6)

Processing dimension d=4...
  Creating loopless nested matroids...
  Creating set compositions...
  Computing stable matroid set compositions...
  Converting to commutative QSym...
  Number of set compositions: 75
  QSym rank: 8 (expected: 8)
  NCQSym rank: 24 (expected: 24)

Summary Table
d   #Matroids    #SetComp     Rank QSym       Rank NCQSym    
--------------------------------------------------------

## Example 3: Conjecture Matrix Computation

This example is based on `matrix_computation_small.py`. We directly compute the conjecture matrix which relates set compositions to loopless nested matroids.

In [6]:
def display_conjecture_matrix(dimension):
    """
    Compute and display the conjecture matrix for a given dimension.
    """
    print(f"\nComputing conjecture matrix for d={dimension}...")
    
    result_matrix, set_comp, matroids = compute_conjecture_matrix(dimension)
    
    print(f"\nResults:")
    print(f"  - Matrix dimensions: {len(result_matrix)} x {len(result_matrix[0]) if result_matrix else 0}")
    print(f"  - Number of set compositions: {len(set_comp)}")
    print(f"  - Number of matroids: {len(matroids)}")
    print(f"  - Matrix rank: {np.linalg.matrix_rank(result_matrix) if result_matrix else 0}")
    
    print(f"\nMatrix (first 5x5):")
    for i, row in enumerate(result_matrix[:min(5, len(result_matrix))]):
        print(f"  [{', '.join(map(str, row[:5]))}{'...' if len(row) > 5 else ''}]")
    
    return result_matrix, set_comp, matroids

# Compute for dimensions 2 and 3
print("=" * 60)
print("CONJECTURE MATRIX COMPUTATION")
print("=" * 60)

for dim in [2, 3]:
    mat, sc, m = display_conjecture_matrix(dim)

CONJECTURE MATRIX COMPUTATION

Computing conjecture matrix for d=2...

Results:
  - Matrix dimensions: 2 x 2
  - Number of set compositions: 2
  - Number of matroids: 2
  - Matrix rank: 2

Matrix (first 5x5):
  [0, 1]
  [1, 1]

Computing conjecture matrix for d=3...

Results:
  - Matrix dimensions: 6 x 6
  - Number of set compositions: 6
  - Number of matroids: 6
  - Matrix rank: 6

Matrix (first 5x5):
  [0, 1, 0, 1, 0...]
  [0, 0, 1, 0, 1...]
  [1, 1, 1, 1, 1...]
  [0, 1, 1, 0, 1...]
  [0, 0, 0, 1, 1...]


## Example 4: Lower Bound Matrix Analysis

In [7]:
def analyze_lowerbound_matrix(dimension):
    """
    Compute and analyze the lowerbound matrix.
    """
    print(f"\nComputing lowerbound matrix for d={dimension}...")
    
    lb_matrix = compute_lowerbound_matrix(dimension)
    
    print(f"  - Matrix dimensions: {len(lb_matrix)} x {len(lb_matrix[0]) if lb_matrix else 0}")
    print(f"  - Matrix rank: {np.linalg.matrix_rank(lb_matrix) if lb_matrix else 0}")
    print(f"  - Matrix (first 4x4):")
    
    for i, row in enumerate(lb_matrix[:min(4, len(lb_matrix))]):
        print(f"    {row[:min(4, len(row))]}")
    
    return lb_matrix

print("\n" + "=" * 60)
print("LOWERBOUND MATRIX ANALYSIS")
print("=" * 60)

for dim in [2, 3]:
    analyze_lowerbound_matrix(dim)


LOWERBOUND MATRIX ANALYSIS

Computing lowerbound matrix for d=2...
  - Matrix dimensions: 2 x 2
  - Matrix rank: 2
  - Matrix (first 4x4):
    [0, 1]
    [1, 1]

Computing lowerbound matrix for d=3...
  - Matrix dimensions: 5 x 5
  - Matrix rank: 5
  - Matrix (first 4x4):
    [0, 1, 1, 1]
    [0, 0, 1, 1]
    [1, 1, 0, 1]
    [1, 1, 0, 0]


## Example 5: Comprehensive Dimension Sweep

This example performs a sweep across multiple dimensions, computing key statistics.

In [ ]:
print("\n" + "=" * 80)
print("COMPREHENSIVE DIMENSION SWEEP")
print("=" * 80)

sweep_results = []

for d in range(2, 9):  # Dimensions 2 through 5
    print(f"\nDimension d={d}")
    print("-" * 80)
    
    # Generate matroids
    loopless_nested = generate_loopless_nested_matroids(d)
    loopless_schubert = generate_loopless_schubert_matroids(d)
    
    # Generate set compositions
    all_sc = SetComposition.generate_all_setcompositions(d)
    
    # Compute NCQSym functions for nested matroids
    print(f"  Computing chromatic functions...")
    nc_functions = [chromatic_non_commutative_quasisymmetric_function(m) 
                   for m in loopless_nested]
    c_functions = [chromatic_quasisymmetric_function(m) 
                  for m in loopless_nested]
    
    # Compute matrix statistics
    def get_matrix_stats(sym_functions):
        all_keys = set()
        for func in sym_functions:
            all_keys.update(func.coefficients.keys())
        
        matrix = [[func.coefficients.get(key, 0) for key in sorted(all_keys)] 
                  for func in sym_functions]
        
        return {
            'rows': len(matrix),
            'cols': len(sorted(all_keys)),
            'rank': np.linalg.matrix_rank(matrix) if matrix else 0
        }
    
    nc_stats = get_matrix_stats(nc_functions)
    c_stats = get_matrix_stats(c_functions)
    
    print(f"  Loopless nested matroids: {len(loopless_nested)}")
    print(f"  Loopless Schubert matroids: {len(loopless_schubert)}")
    print(f"  Set compositions: {len(all_sc)}")
    print(f"  NCQSym matrix: {nc_stats['rows']}x{nc_stats['cols']}, rank={nc_stats['rank']}")
    print(f"  QSym matrix: {c_stats['rows']}x{c_stats['cols']}, rank={c_stats['rank']}")
    
    sweep_results.append({
        'd': d,
        'nested': len(loopless_nested),
        'schubert': len(loopless_schubert),
        'set_comp': len(all_sc),
        'nc_rank': nc_stats['rank'],
        'c_rank': c_stats['rank']
    })

print("\n" + "=" * 80)
print("Summary")
print("=" * 80)
print(f"{'d':<3} {'Nested':<8} {'Schubert':<10} {'SetComp':<8} {'NCQSym Rank':<12} {'QSym Rank':<12}")
print("-" * 80)
for r in sweep_results:
    print(f"{r['d']:<3} {r['nested']:<8} {r['schubert']:<10} {r['set_comp']:<8} "
          f"{r['nc_rank']:<12} {r['c_rank']:<12}")


COMPREHENSIVE DIMENSION SWEEP

Dimension d=2
--------------------------------------------------------------------------------
  Computing chromatic functions...
  Loopless nested matroids: 2
  Loopless Schubert matroids: 2
  Set compositions: 3
  NCQSym matrix: 2x3, rank=2
  QSym matrix: 2x2, rank=2

Dimension d=3
--------------------------------------------------------------------------------
  Computing chromatic functions...
  Loopless nested matroids: 6
  Loopless Schubert matroids: 4
  Set compositions: 13
  NCQSym matrix: 6x13, rank=6
  QSym matrix: 6x4, rank=4

Dimension d=4
--------------------------------------------------------------------------------
  Computing chromatic functions...
  Loopless nested matroids: 24
  Loopless Schubert matroids: 8
  Set compositions: 75
  NCQSym matrix: 24x75, rank=24
  QSym matrix: 24x8, rank=8

Dimension d=5
--------------------------------------------------------------------------------
  Computing chromatic functions...
  Loopless nested

## Summary

This notebook demonstrated practical matrix computations using the chromatic matroids package:

1. **Basic Rank Computation**: Computing chromatic NCQSym functions and analyzing matrix rank
2. **Dimension Analysis**: Comparing QSym and NCQSym ranks across dimensions
3. **Conjecture Matrix**: Direct computation of conjecture matrices
4. **Lowerbound Matrix**: Analysis of lowerbound matrices
5. **Comprehensive Sweep**: Multi-dimensional analysis of matroid families

These examples showcase how the package can be used for research in algebraic combinatorics, particularly for studying relationships between matroids and quasisymmetric functions.